# 经典轨迹滤波链演示

本示例通过一个虚构的航班轨迹，逐步演示 `filterclassic.py` 中的滤波器如何协同工作：

1. **FilterCstLatLon**：识别重复广播的经纬度。
2. **FilterCstPosition**：三维位置整体未更新时清空。
3. **FilterCstSpeed**：速度相关字段未更新时清空。
4. **MyFilterDerivative**：利用一阶/二阶导的阈值剔除突刺点。
5. **FilterIsolated**：屏蔽与其它点相距 ≥20 秒的孤立观测。

> 说明：示例数据极简且专为演示设计，真实项目中请使用 `filter_trajs.py` 通过 `traffic` 的流水线调用这些滤波器。


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

from filterclassic import (
    FilterCstLatLon,
    FilterCstPosition,
    FilterCstSpeed,
    FilterIsolated,
    MyFilterDerivative,
)


In [ ]:
base_time = pd.Timestamp('2024-01-01 00:00:00')
timestamps = base_time + pd.to_timedelta([0, 5, 10, 12, 14, 40], unit='s')

raw_df = pd.DataFrame(
    {
        'timestamp': timestamps,
        'icao24': ['abcdef'] * len(timestamps),
        'flight_id': [1234] * len(timestamps),
        'latitude': [30.0000, 30.0000, 30.0020, 30.0020, 30.0500, 30.0600],
        'longitude': [120.0000, 120.0000, 120.0030, 120.0030, 120.0700, 120.0800],
        'altitude': [10000, 10000, 10050, 13050, 10060, 10070],
        'vertical_rate': [0, 0, 0, -3000, 0, 0],
        'track': [90, 90, 90, 100, 95, 95],
        'groundspeed': [200, 200, 200, 220, 205, 205],
        'u_component_of_wind': [5, 5, 6, 6, 7, 8],
        'v_component_of_wind': [-2, -2, -1, -1, 0, 0],
        'temperature': [270, 270, 268, 268, 266, 265],
    }
)
print('原始数据：')
display(raw_df)


In [ ]:
filters = [
    ('FilterCstLatLon', FilterCstLatLon()),
    ('FilterCstPosition', FilterCstPosition()),
    ('FilterCstSpeed', FilterCstSpeed()),
    ('MyFilterDerivative', MyFilterDerivative()),
    ('FilterIsolated', FilterIsolated()),
]

current = raw_df.copy()
for name, flt in filters:
    current = flt.apply(current)
    print(f'应用 {name} 之后：')
    display(current[['timestamp', 'latitude', 'longitude', 'altitude', 'vertical_rate', 'track', 'groundspeed']])


In [ ]:
print('演示变量联动屏蔽：')
processed = raw_df.copy()
for name, flt in filters:
    processed = flt.apply(processed)
mask_map = {
    'latitude': ['u_component_of_wind', 'v_component_of_wind', 'temperature'],
    'altitude': ['u_component_of_wind', 'v_component_of_wind', 'temperature'],
}
for key, cols in mask_map.items():
    for col in cols:
        processed[col] = processed[[col]].mask(processed[key].isna())
display(processed[['timestamp', 'latitude', 'altitude', 'u_component_of_wind', 'temperature']])


## 小结

- 连续相同的经纬度或高度会被识别并置 NaN。
- 一次性的大幅高度突刺会被导数滤波器捕获并清除。
- 与其它点相距超过 20 秒的孤立观测会被 `FilterIsolated` 标记为空值。
- 当位置/高度缺失时，同步屏蔽天气变量，避免后续模型误用。

可结合 `filter_trajs.py` 的 CLI 接口，在真实数据上重现同样的处理流程。
